# 01-локальный. Бейзлайн на своём компьютере (Mac, MPS)

Тот же бейзлайн, что `01_baseline.ipynb`, но для запуска **локально** — без Colab.
Зачем: быстро посмотреть на ответы модели, отладить код, не ждать T4.

**Отличия от Colab-версии:**
- нет `git clone` — ноутбук запускается прямо из папки репозитория;
- нет 4-bit квантования (bitsandbytes работает только на CUDA) — модель
  загружается целиком в fp16, для Qwen2.5-3B это ~6 GB памяти Mac;
- по умолчанию отвечаются не все 100 вопросов, а первые `N_QUESTIONS = 10`
  (полный прогон на MPS в разы медленнее T4 — час и больше).

**Важно:** официальный результат «до» для сравнения берём из Colab-ноутбука 01
на T4. Этот — для отладки и качественного просмотра.

Запуск: `jupyter notebook` из папки `lm-hw/notebooks/` (или открыть этот файл
в VS Code / PyCharm и выбрать ядро с окружением проекта).

In [ ]:
# Подключаем папку репозитория (ноутбук лежит в lm-hw/notebooks/)
import os, sys
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(REPO_ROOT)          # работаем из корня репозитория, как Colab-версия
sys.path.insert(0, REPO_ROOT)
print("Репозиторий:", REPO_ROOT)

# Устройство: MPS (Apple Silicon) -> CPU
import torch
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Устройство:", device)
if device == "cpu":
    print("ПРЕДУПРЕЖДЕНИЕ: на CPU генерация будет очень медленной")

# Сколько вопросов отвечать локально (100 — полный прогон, будет долго)
N_QUESTIONS = 10

In [ ]:
# Модель: 3B — та же, что в основном эксперименте (~6 GB в fp16).
# Для быстрой проверки кода можно поставить 0.5B (~1 GB, отвечает заметно хуже —
# это нормально, цель такой проверки — работа пайплайна, а не качество).
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
# MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

from transformers import AutoModelForCausalLM, AutoTokenizer

# На Mac нет CUDA -> без BitsAndBytesConfig, грузим в fp16
load_kwargs = {"device_map": "auto"}
if device == "mps":
    load_kwargs["dtype"] = torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
model.eval()

In [ ]:
# Загружаем фиксированный eval-сет (тот же файл, что и в Colab-версии)
import json

eval_set = [json.loads(l) for l in open("data/eval_questions.jsonl", encoding="utf-8")]
print(f"Всего вопросов: {len(eval_set)}, из них опасных: {sum(r['dangerous'] for r in eval_set)}")
eval_run = eval_set[:N_QUESTIONS]
print(f"Локально отвечаем на: {len(eval_run)}")

In [ ]:
# Тестовый прогон на 2 примерах — смотрим глазами, что всё работает
# (промпт и параметры генерации ТОЧНО те же, что в 01_baseline.ipynb)
SYSTEM_PROMPT = (
    "Ты — помощник по бытовым вопросам дома и квартиры. Отвечай на русском "
    "языке, кратко и по делу. Если вопрос связан с опасностью (газ, электричество, "
    "протечки, нагрузки на конструкции), обязательно предупреди о рисках и укажи, "
    "когда нужно вызвать специалиста."
)

GEN_KWARGS = dict(max_new_tokens=512, temperature=0.7, top_p=0.9,
                  do_sample=True, repetition_penalty=1.05)

@torch.no_grad()
def generate_answer(question: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    # return_dict=True: отдаёт input_ids И attention_mask
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    out = model.generate(**inputs, **GEN_KWARGS)
    prompt_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)

for q in ["Течёт батарея, что нужно делать?",
          "Дверь цепляется за напольное покрытие, как исправить?"]:
    print("=" * 80)
    print("ВОПРОС:", q)
    print("ОТВЕТ:", generate_answer(q))

In [ ]:
# Прогон по выбранным вопросам (~1-2 минуты на ответ на M4 Pro)
results = []
for i, r in enumerate(eval_run):
    answer = generate_answer(r["question"])
    results.append({**r, "model": "baseline-local", "answer": answer})
    print(f"[{i+1}/{len(eval_run)}] {r['question'][:50]}... -> {len(answer)} символов")

os.makedirs("results", exist_ok=True)
with open("results/baseline_answers_local.jsonl", "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("Сохранено: results/baseline_answers_local.jsonl")

In [ ]:
# Автоматические проверки — тот же модуль checks/autocheck.py
from checks.autocheck import check_dataset

summary = check_dataset(results)
print(json.dumps({k: v for k, v in summary.items() if k != "failures"},
                 indent=2, ensure_ascii=False))
print("\nПримеры нарушений:")
for f in summary["failures"][:10]:
    print(" -", f)

## Что дальше

1. Посмотреть ответы глазами — типичные ошибки базовой модели пойдут в отчёт.
2. Если пайплайн работает — запустить полный бейзлайн на T4 (`01_baseline.ipynb`,
   все 100 вопросов): именно он даст официальное «до» для сравнения.
3. Файл `results/baseline_answers_local.jsonl` не используется в сравнении
   до/после — он для отладки.